# Retriever Comparison: Dense vs Hybrid vs Ensemble

이 노트북은 Dense, Hybrid, Ensemble Retriever의 검색 성능을 정량적으로 비교(Recall, NDCG)하고 분석하기 위해 작성되었습니다.

In [14]:
import sys
import os
from dotenv import load_dotenv, find_dotenv
import pandas as pd
import math

# Add parent directory to path to import app modules
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

# Load env vars
load_dotenv(find_dotenv())

try:
    from app.db.vector_db import (
        get_dense_retriever,
        get_hybrid_retriever,
        get_ensemble_retriever,
    )
    print("Successfully imported retrievers.")
except ImportError as e:
    print(f"Error importing modules: {e}")
    print("Please make sure you are running this notebook from the 'lab' directory and imports are correct.")

Successfully imported retrievers.


## 1. Evaluation Metrics (Recall@K, NDCG@K) using scikit-learn

In [15]:
# Install scikit-learn if not already installed
try:
    import sklearn
    from sklearn.metrics import ndcg_score, recall_score
    import numpy as np
    print("scikit-learn is ready.")
except ImportError:
    print("scikit-learn not found. Installing...")
    import sys
    %{sys.executable} -m pip install scikit-learn
    from sklearn.metrics import ndcg_score, recall_score
    import numpy as np
    print("Installed scikit-learn.")

def calculate_metrics_using_sklearn(retrieved_ids, true_ids, k=5):
    """
    Calculate Recall@K and NDCG@K using scikit-learn.
    retrieved_ids: list of strings (document IDs retrieved)
    true_ids: list of strings (relevant document IDs)
    k: int
    """
    if not true_ids:
        return 0.0, 0.0
    
    # 1. universe construction: Union of Retrieved and True IDs
    all_ids = list(set(retrieved_ids) | set(true_ids))
    id_to_index = {id_: i for i, id_ in enumerate(all_ids)}
    n_items = len(all_ids)
    
    # 2. y_true (Ground Truth)
    y_true = np.zeros(n_items)
    for tid in true_ids:
        y_true[id_to_index[tid]] = 1
        
    # 3. y_score (Predicted Scores)
    # We simulate scores based on rank. 
    # First retrieved item gets highest score.
    y_score = np.zeros(n_items)
    score_val = len(retrieved_ids) + 1.0
    for rid in retrieved_ids:
        if rid in id_to_index:
            y_score[id_to_index[rid]] = score_val
            score_val -= 1.0
            
    # Reshape for sklearn input (n_samples=1, n_labels)
    y_true_reshaped = y_true.reshape(1, -1)
    y_score_reshaped = y_score.reshape(1, -1)
    
    # Calculate NDCG@K
    ndcg = ndcg_score(y_true_reshaped, y_score_reshaped, k=k)
    
    # Calculate Recall@K
    # Construct y_pred_binary for top K retrieved items
    top_k_ids = retrieved_ids[:min(k, len(retrieved_ids))]
    y_pred_binary = np.zeros(n_items)
    for rid in top_k_ids:
        if rid in id_to_index:
            y_pred_binary[id_to_index[rid]] = 1
            
    recall = recall_score(y_true, y_pred_binary)
    
    return recall, ndcg

scikit-learn is ready.


## 2. Initialize Retrievers

In [16]:
# Initialize retrievers with K=10 (getting more candidates for re-ranking simulation or broad checking)
# You can adjust K as needed
K_RETRIEVAL = 10

try:
    dense_retriever = get_dense_retriever(k=K_RETRIEVAL)
    hybrid_retriever = get_hybrid_retriever(k=K_RETRIEVAL)
    ensemble_retriever = get_ensemble_retriever(k=K_RETRIEVAL, dense_weight=0.7, sparse_weight=0.3)
    print("Retrievers initialized successfully.")
except Exception as e:
    print(f"Failed to initialize retrievers: {e}")

Retrievers initialized successfully.


## 3. Ground Truth Data Setup
**중요**: 여기에 테스트할 쿼리와 예상되는 정답 문서의 ID(`no`)를 입력해주세요.
DB를 조회하여 실제 존재하는 `no`를 넣어야 정확한 평가가 가능합니다.

In [17]:
# Format: {"query": ["expected_doc_id_1", "expected_doc_id_2", ...]}
# 예시 데이터 - 실제 ID로 교체 필요
ground_truth = {
    # 예시: "해운대": ["126081", "..."],
    "야경이 이쁜 곳": [
        "56696", "56745", "56755", "56777", "56816", "56834", "56835", "56847", "56888", "56890", "56895", "56977", "57052", "57142", "57236", "57254", "57263", "57268", "57345", "57369", "57402", "57456", "57461", "57527", "57573", "57605", "57640", "57669", "57691", "57734", "57737", "57757", "57798", "57825", "57892", "57904", "57941", "57965", "58004", "58009", "58031", "58081", "58094", "58209", "58296", "58298", "58311", "58338", "58486", "58606", "58619", "58660", "58665", "58666", "58678", "58686", "58693", "58711", "58724", "58747", "58750", "58754", "58761", "58764", "58774", "58783", "58802", "58817", "58823", "58830", "58840", "58842", "58859", "58884", "58887", "58899", "58904", "58915", "58932", "58946", "58947", "58986", "59036", "59043", "59135", "59148", "59165", "59861", "59883", "60698", "60855", "62014", "62636", "62661", "63124", "63223", "63344", "63519", "63683", "63859", "64035", "64112", "64600", "64846", "65131", "65190", "65344", "65473", "65635", "65657", "65724", "65802", "66077", "66297", "66351", "66357", "66474", "66524", "66595", "66596", "66629", "66634", "66642", "66882", "67542", "67648", "67861", "67893", "67902", "68083", "68142", "68178", "68189", "68190", "68191", "68229", "68265", "68507", "68632", "68634", "68646", "68747", "68830", "68941", "69092", "69229", "69257", "69736", "70745", "72571", "72591", "72606", "72741", "72743", "72754", "72763", "72796", "72813", "72838", "72842", "72868", "72941", "72955", "72982", "73006", "73027", "73048", "73064", "73084", "73095", "73097", "73109", "73112", "73205", "73212", "73255", "73256", "73322", "73378", "73538", "73580", "73625", "73646", "73700", "73772", "73786", "73807", "73828", "73854", "73861", "73902", "73936", "73980", "74031", "74032", "74068", "74070", "74094", "74174", "74180", "74195", "74239", "74240", "74259", "74304", "74306", "74401", "74458", "74695", "74719", "74839", "74863", "74876", "74963", "75060", "75143", "75170", "75189", "75290", "75364", "75634", "75635", "75636", "75657", "75680", "75696", "75701", "75708", "75722", "75750", "76371", "76806", "78423", "78557", "79554", "79836", "79839", "79841", "79979", "79986", "80049", "80079", "80094", "80315", "80350", "80382", "80403", "80445", "80487", "80522", "80564", "80566", "80567", "80576", "80582", "80650", "81907", "81953", "82095", "82596", "82619", "82781", "82953", "83001", "83021", "83089", "83092", "83121", "83556", "83577", "83585", "83640", "83719", "83854", "83909", "84016", "85077", "85372", "85510", "85545", "85926", "86490", "86655", "86718", "86888", "86896", "87309", "87613", "87764", "87843", "90018", "90556", "91461", "91777", "91897", "92022", "92046", "92061", "92114", "92119", "92147", "92220", "92354", "92362", "92454", "92480", "92489", "92855", "93103", "93119", "93192", "93213", "93270", "93303", "93337", "93375", "93395", "93527", "93548", "93567", "93613", "93617", "93647", "93691", "93697", "93738", "93746", "93834", "93868", "93875", "93876", "93962", "93987", "94033", "94044", "94185", "94189", "94391", "94453", "94480", "94509", "94527", "94544", "94563", "94610", "94694", "94869", "94888", "95165", "95368", "95387", "95426", "95637", "95725", "95734", "95751", "95761", "95793", "95932", "95981", "96041", "96119", "96133", "96418", "97460", "99406", "99656", "100691", "101697", "101977", "103874", "104478", "104931", "105857", "105882", "105980", "106141", "106166", "106279", "106453", "106475", "106608", "106612", "106633"
    ],
    "아이와 함께 가기 좋은 곳": ["56971", "57116", "57358", "57485", "57675", "57909", "58402", "58656", "58691", "59009", "59085", "59323", "59628", "59834", "60042", "60096", "60233", "60338", "60502", "60609", "60765", "60782", "61149", "61299", "61546", "61576", "61613", "61726", "61874", "62285", "63820", "64150", "65105", "65480", "66259", "66492", "67142", "69515", "70163", "70203", "70871", "70890", "70935", "71307", "72363", "72833", "72932", "72937", "73279", "73350", "73833", "74179", "74422", "74960", "75090", "75671", "76151", "76351", "76516", "76689", "78387", "79144", "79702", "80479", "80746", "82436", "84210", "84635", "84841", "85782", "86234", "86404", "86512", "87424", "88031", "88170", "88171", "88196", "88384", "88504", "88526", "88723", "88954", "89389", "89745", "89920", "90107", "91289", "95462", "96179", "96758", "97925", "98774", "102453", "106920"],
    "산책하기 좋은 곳": ["56645", "56646", "56652", "56655", "56659", "56660", "56673", "56674", "56678", "56679", "56687", "56688", "56691", "56696", "56698", "56711", "56719", "56726", "56738", "56753", "56757", "56760", "56761", "56766", "56770", "56771", "56781", "56784", "56785", "56786", "56789", "56793", "56799", "56810", "56816", "56820", "56831", "56845", "56846", "56852", "56853", "56854", "56865", "56867", "56868", "56887", "56889", "56893", "56894", "56895", "56896", "56903", "56911", "56921", "56924", "56926", "56946", "56956", "56971", "56972", "56975", "56984", "56986", "57049", "57054", "57061", "57080", "57097", "57111", "57121", "57125", "57153", "57154", "57159", "57160", "57176", "57178", "57181", "57182", "57188", "57198", "57200", "57201", "57202", "57212", "57214", "57221", "57223", "57231", "57238", "57239", "57241", "57242", "57253", "57254", "57259", "57260", "57271", "57275", "57285", "57287", "57297", "57300", "57303", "57304", "57305", "57317", "57324", "57331", "57344", "57349", "57351", "57357", "57358", "57360", "57361", "57365", "57366", "57367", "57369", "57389", "57394", "57403", "57407", "57410", "57419", "57426", "57430", "57438", "57444", "57449", "57461", "57480", "57488", "57525", "57527", "57528", "57529", "57530", "57532", "57535", "57536", "57537", "57539", "57540", "57543", "57556", "57559", "57560", "57566", "57567", "57578", "57579", "57581", "57584", "57588", "57607", "57612", "57614", "57615", "57620", "57629", "57630", "57634", "57668", "57670", "57671", "57672", "57675", "57678", "57679", "57681", "57683", "57684", "57686", "57687", "57694", "57696", "57697", "57701", "57702", "57705", "57707", "57708", "57710", "57713", "57714", "57721", "57722", "57723", "57724", "57734", "57737", "57739", "57749", "57754", "57758", "57759", "57766", "57769", "57780", "57790", "57793", "57802", "57822", "57827", "57834", "57836", "57848", "57851", "57852", "57853", "57858", "57859", "57860", "57862", "57872", "57891", "57892", "57897", "57900", "57911", "57915", "57920", "57926", "57935", "57943", "57944", "57946", "57948", "57950", "57951", "57952", "57954", "57959", "57960", "57961", "57963", "57964", "57969", "57986", "57991", "57998", "58004", "58007", "58008", "58013", "58018", "58020", "58021", "58022", "58029", "58032", "58034", "58043", "58048", "58051", "58052", "58055", "58056", "58058", "58060", "58061", "58063", "58067", "58069", "58072", "58073", "58082", "58084", "58085", "58090", "58095", "58098", "58100", "58103", "58118", "58125", "58127", "58131", "58138", "58140", "58150", "58151", "58159", "58160", "58163", "58164", "58174", "58177", "58179", "58190", "58194", "58197", "58204", "58214", "58217", "58218", "58227", "58228", "58236", "58239", "58254", "58256", "58257", "58258", "58267", "58268", "58271", "58276", "58277", "58278", "58281", "58282", "58284", "58286", "58287", "58295", "58296", "58305", "58306", "58313", "58314", "58315", "58321", "58322", "58324", "58325", "58326", "58327", "58329", "58330", "58332", "58333", "58334", "58340", "58343", "58355", "58367", "58370", "58376", "58385", "58391", "58394", "58399", "58400", "58401", "58403", "58408", "58409", "58418", "58428", "58430", "58432", "58434", "58440", "58443", "58444", "58445", "58454", "58455", "58466", "58471", "58476", "58479", "58485", "58488", "58490", "58499", "58502", "58503", "58510", "58516", "58517", "58527", "58528", "58536", "58560", "58561", "58575", "58577", "58579", "58582", "58585", "58586", "58590", "58599", "58600", "58601", "58602", "58603", "58604", "58605", "58606", "58607", "58610", "58612", "58618", "58619", "58632", "58639", "58657", "58660", "58661", "58664", "58669", "58670", "58672", "58673", "58682", "58683", "58685", "58686", "58690", "58695", "58696", "58710", "58711", "58712", "58715", "58719", "58729", "58733", "58734", "58742", "58761", "58765", "58770", "58772", "58773", "58774", "58781", "58782", "58783", "58785", "58791", "58793", "58796", "58797", "58799", "58806", "58808", "58818", "58821", "58822", "58823", "58826", "58829", "58832", "58833", "58836", "58841", "58842", "58843", "58845", "58851", "58854", "58863", "58872", "58873", "58884", "58885", "58886", "58891", "58893", "58894", "58917", "58920", "58922", "58926", "58928", "58929", "58930", "58939", "58942", "58944", "58945", "58949", "58959", "58963", "58965", "58969", "58970", "58979", "58982", "58983", "58986", "58991", "58992", "58999", "59006", "59015", "59016", "59025", "59032", "59036", "59037", "59038", "59040", "59041", "59042", "59043", "59044", "59048", "59053", "59055", "59060", "59061", "59094", "59096", "59104", "59107", "59120", "59123", "59135", "59137", "59139", "59141", "59145", "59153", "59154", "59161", "59162", "59163", "59165", "59173", "59176", "59177", "59178", "59180", "59191", "59193", "59202", "59204", "59205", "59209", "59212", "59233", "59234", "59235", "59237", "59238", "59247", "59250", "59251", "59267", "59268", "59277", "59282", "59283", "59287", "59288", "59297", "59304", "59306", "59308", "59310", "59311", "59312", "59314", "59315", "59316", "59317", "59318", "59323", "59326", "59327", "59331", "59332", "59333", "59334", "59344", "59346", "59354", "59356", "59363", "59376", "59379", "59380", "59381", "59382", "59386", "59393", "59395", "59396", "59400", "59401", "59403", "59404", "59408", "59410", "59411", "59413", "59416", "59417", "59420", "59431", "59435", "59437", "59438", "59440", "59445", "59447", "59448", "59450", "59455", "59456", "59459", "59460", "59462", "59463", "59466", "59469", "59470", "59472", "59482", "59485", "59488", "59490", "59497", "59499", "59508", "59513", "59516", "59518", "59519", "59525", "59534", "59536", "59541", "59543", "59544", "59545", "59551", "59556", "59558", "59559", "59564", "59569", "59575", "59577", "59579", "59580", "59581", "59582", "59584", "59586", "59587", "59596", "59597", "59603", "59604", "59606", "59609", "59610", "59611", "59615", "59616", "59621", "59623", "59624", "59628", "59630", "59635", "59636", "59644", "59647", "59662", "59665", "59667", "59668", "59671", "59676", "59678", "59679", "59683", "59686", "59687", "59688", "59689", "59690", "59692", "59697", "59704", "59708", "59709", "59713", "59715", "59716", "59719", "59723", "59724", "59728", "59733", "59735", "59738", "59739", "59740", "59743", "59749", "59760", "59763", "59765", "59769", "59773", "59776", "59786", "59789", "59795", "59796", "59797", "59798", "59800", "59801", "59806", "59812", "59816", "59817", "59818", "59821", "59825", "59832", "59835", "59838", "59842", "59850", "59851", "59853", "59858", "59859", "59861", "59862", "59866", "59870", "59871", "59874", "59878", "59880", "59886", "59887", "59889", "59890", "59893", "59896", "59898", "59903", "59904", "59909", "59912", "59913", "59914", "59917", "59919", "59921", "59922", "59924", "59925", "59926", "59927", "59928", "59929", "59938", "59946", "59949", "59962", "59974", "59981", "59982", "59983", "59984", "59988", "59989", "59990", "59991", "60005", "60006", "60007", "60012", "60013", "60014", "60018", "60019", "60020", "60022", "60023", "60027", "60030", "60036", "60040", "60052", "60054", "60061", "60062", "60065", "60068", "60073", "60074", "60075", "60079", "60084", "60086", "60087", "60089", "60090", "60091", "60092", "60104", "60107", "60109", "60110", "60112", "60125", "60126", "60130", "60139", "60141", "60151", "60153", "60158", "60160", "60161", "60169", "60175", "60177", "60178", "60204", "60207", "60209", "60210", "60213", "60216", "60217", "60218", "60224", "60226", "60227", "60231", "60236", "60241", "60246", "60247", "60250", "60252", "60253", "60268", "60270", "60272", "60273", "60274", "60280", "60286", "60291", "60295", "60296", "60298", "60308", "60314", "60317", "60321", "60325", "60326", "60329", "60333", "60343", "60344", "60346", "60348", "60356", "60357", "60360", "60362", "60366", "60368", "60375", "60379", "60391", "60392", "60402", "60405", "60406", "60410", "60411", "60414", "60415", "60416", "60418", "60419", "60434", "60438", "60441", "60446", "60447", "60448", "60449", "60450", "60460", "60461", "60466", "60469", "60470", "60471", "60472", "60479", "60482", "60487", "60490", "60501", "60504", "60506", "60507", "60509", "60510", "60512", "60513", "60516", "60523", "60524", "60530", "60533", "60534", "60535", "60537", "60543", "60547", "60549", "60550", "60553", "60554", "60559", "60563", "60564", "60571", "60576", "60577", "60578", "60579", "60582", "60583", "60586", "60587", "60590", "60591", "60595", "60602", "60616", "60621", "60622", "60626", "60643", "60645", "60658", "60666", "60671", "60679", "60680", "60682", "60686", "60695", "60697", "60707", "60732", "60735", "60737", "60738", "60741", "60743", "60745", "60746", "60756", "60761", "60771", "60774", "60779", "60783", "60785", "60797", "60799", "60801", "60802", "60805", "60806", "60810", "60817", "60819", "60827", "60829", "60832", "60835", "60836", "60841", "60843", "60849", "60850", "60854", "60861", "60872", "60881", "60883", "60888", "60889", "60891", "60896", "60906", "60908", "60911", "60924", "60925", "60929", "60934", "60936", "60941", "60942", "60944", "60947", "60950", "60956", "60957", "60958", "60961", "60965", "60970", "60976", "60977", "60985", "60996", "60997", "61003", "61011", "61013", "61017", "61031", "61039", "61042", "61046", "61048", "61053", "61057", "61064", "61065", "61068", "61088", "61102", "61104", "61107", "61108", "61119", "61124", "61141", "61150", "61174", "61176", "61183", "61185", "61190", "61193", "61197", "61200", "61201", "61204", "61212", "61213", "61217", "61220", "61225", "61242", "61250", "61262", "61267", "61269", "61274", "61279", "61280", "61301", "61303", "61307", "61308", "61310", "61311", "61316", "61319", "61332", "61344", "61361", "61370", "61371", "61395", "61400", "61408", "61409", "61410", "61411", "61422", "61425", "61432", "61436", "61437", "61450", "61459", "61467", "61477", "61478", "61479", "61493", "61501", "61503", "61523", "61531", "61536", "61542", "61546", "61549", "61555", "61565", "61567", "61578", "61582", "61584", "61589", "61593", "61606", "61609", "61610", "61611", "61612", "61614", "61618", "61634", "61636", "61643", "61647", "61661", "61665", "61672", "61674", "61679", "61686", "61689", "61691", "61696", "61704", "61709", "61711", "61714", "61733", "61735", "61736", "61746", "61767", "61768", "61771", "61775", "61781", "61794", "61796", "61806", "61809", "61823", "61824", "61832", "61846", "61851", "61854", "61863", "61864", "61872", "61873", "61881", "61883", "61884", "61886", "61894", "61895", "61904", "61916", "61918", "61927", "61943", "61945", "61946", "61947", "61948", "61957", "61961", "61969", "61972", "61994", "61998", "62012", "62013", "62014", "62015", "62021", "62030", "62031", "62033", "62044", "62045", "62048", "62049", "62055", "62056", "62058", "62066", "62072", "62085", "62100", "62115", "62117", "62127", "62128", "62132", "62154", "62155", "62158", "62169", "62170", "62171", "62173", "62181", "62192", "62194", "62197", "62212", "62233", "62235", "62239", "62242", "62255", "62267", "62269", "62281", "62311", "62317", "62318", "62319", "62322", "62329", "62331", "62339", "62342", "62348", "62356", "62357", "62361", "62364", "62367", "62369", "62370", "62373", "62374", "62375", "62379", "62384", "62386", "62387", "62392", "62395", "62398", "62400", "62408", "62410", "62412", "62417", "62423", "62424", "62427", "62432", "62435", "62439", "62447", "62448", "62456", "62459", "62464", "62470", "62471", "62474", "62477", "62478", "62481", "62482", "62489", "62490", "62491", "62497", "62511", "62516", "62517", "62518", "62526", "62529", "62530", "62531", "62534", "62548", "62552", "62555", "62559", "62565", "62567", "62568", "62569", "62570", "62572", "62573", "62574", "62577", "62578", "62579", "62591", "62597", "62598", "62603", "62607", "62608", "62609", "62610", "62612", "62613", "62616", "62620", "62621", "62622", "62625", "62626", "62627", "62629", "62631", "62634", "62657", "62660", "62661", "62662", "62666", "62671", "62673", "62677", "62679", "62689", "62690", "62703", "62706", "62710", "62711", "62713", "62714", "62715", "62720", "62721", "62724", "62729", "62734", "62735", "62737", "62738", "62743", "62747", "62750", "62754", "62755", "62759", "62763", "62766", "62769", "62773", "62775", "62777", "62778", "62779", "62780", "62801", "62809", "62814", "62822", "62823", "62824", "62829", "62831", "62837", "62839", "62840", "62841", "62842", "62843", "62846", "62853", "62856", "62866", "62867", "62868", "62870", "62880", "62886", "62890", "62892", "62899", "62900", "62901", "62903", "62904", "62912", "62917", "62920", "62921", "62925", "62927", "62932", "62934", "62937", "62938", "62941", "62943", "62946", "62954", "62956", "62970", "62989", "62990", "62992", "62995", "62998", "62999", "63001", "63002", "63003", "63008", "63009", "63022", "63024", "63027", "63030", "63032", "63033", "63036", "63037", "63039", "63040", "63042", "63045", "63046", "63050", "63072", "63080", "63083", "63087", "63088", "63095", "63100", "63102", "63104", "63112", "63113", "63115", "63116", "63118", "63121", "63126", "63135", "63149", "63158", "63164", "63165", "63166", "63169", "63180", "63181", "63188", "63196", "63197", "63201", "63202", "63214", "63215", "63216", "63217", "63229", "63236", "63237", "63242", "63246", "63248", "63253", "63254", "63264", "63268", "63270", "63291", "63315", "63323", "63325", "63331", "63338", "63340", "63352", "63360", "63364", "63367", "63378", "63381", "63382", "63383", "63385", "63389", "63390", "63392", "63401", "63403", "63406", "63409", "63418", "63431", "63433", "63437", "63442", "63445", "63448", "63453", "63463", "63466", "63474", "63475", "63476", "63480", "63486", "63487", "63494", "63495", "63510", "63514", "63521", "63527", "63536", "63539", "63543", "63560", "63561", "63569", "63570", "63571", "63578", "63584", "63587", "63588", "63589", "63595", "63604", "63607", "63620", "63628", "63635", "63636", "63637", "63640", "63641", "63646", "63650", "63656", "63659", "63671", "63677", "63680", "63682", "63688", "63693", "63699", "63729", "63734", "63747", "63748", "63760", "63768", "63785", "63787", "63790", "63794", "63795", "63798", "63801", "63804", "63817", "63818", "63822", "63835", "63842", "63854", "63860", "63862", "63867", "63874", "63890", "63898", "63902", "63906", "63911", "63918", "63920", "63921", "63922", "63944", "63952", "63953", "63964", "63968", "63985", "63986", "63993", "63996", "64002", "64019", "64029", "64033", "64038", "64049", "64051", "64057", "64058", "64065", "64069", "64079", "64085", "64092", "64096", "64096", "64106", "64110", "64130", "64131", "64138", "64140", "64141", "64146", "64148", "64149", "64156", "64168", "64169", "64183", "64186", "64194", "64205", "64207", "64210", "64217", "64225", "64235", "64243", "64244", "64258", "64267", "64268", "64270", "64272", "64274", "64276", "64281", "64282", "64284", "64285", "64291", "64296", "64299", "64309", "64315", "64317", "64318", "64322", "64325", "64326", "64327", "64329", "64340", "64347", "64349", "64360", "64365", "64382", "64389", "64391", "64394", "64396", "64397", "64399", "64400", "64402", "64405", "64425", "64426", "64433", "64434", "64435", "64436", "64449", "64474", "64477", "64479", "64480", "64489", "64495", "64496", "64498", "64509", "64515", "64520", "64522", "64538", "64551", "64552", "64563", "64580", "64591", "64598", "64600", "64604", "64606", "64613", "64614", "64616", "64626", "64628", "64635", "64646", "64652", "64666", "64684", "64692", "64701", "64702", "64704", "64706", "64713", "64714", "64726", "64743", "64747", "64750", "64753", "64773", "64774", "64787", "64790", "64803", "64816", "64829", "64832", "64845", "64864", "64866", "64910", "64916", "64918", "64922", "64933", "64937", "64941", "64949", "64954", "64963", "64964", "64968", "64975", "64983", "64988", "64992", "64994", "64999", "65003", "65005", "65013", "65014", "65020", "65021", "65029", "65031", "65036", "65038", "65066", "65067", "65094", "65095", "65096", "65098", "65100", "65102", "65109", "65112", "65120", "65121", "65129", "65134", "65137", "65143", "65152", "65170", "65173", "65174", "65179", "65198", "65199", "65200", "65209", "65213", "65225", "65244", "65247", "65249", "65252", "65256", "65261", "65263", "65264", "65267", "65269", "65276", "65277", "65283", "65286", "65295", "65307", "65308", "65319", "65327", "65330", "65331", "65334", "65337", "65339", "65340", "65346", "65361", "65363", "65371", "65377", "65379", "65385", "65386", "65389", "65392", "65394", "65403", "65404", "65407", "65414", "65417", "65418", "65419", "65420", "65425", "65432", "65433", "65436", "65442", "65444", "65449", "65452", "65455", "65465", "65469", "65471", "65476", "65478", "65497", "65501", "65502", "65509", "65516", "65524", "65527", "65535", "65540", "65542", "65559", "65562", "65564", "65565", "65568", "65572", "65588", "65590", "65601", "65615", "65618", "65621", "65632", "65633", "65636", "65637", "65643", "65649", "65654", "65656", "65657", "65673", "65676", "65683", "65692", "65701", "65702", "65705", "65709", "65721", "65722", "65725", "65730", "65733", "65739", "65743", "65757", "65766", "65769", "65770", "65775", "65778", "65781", "65792", "65812", "65819", "65820", "65823", "65828", "65833", "65834", "65836", "65843", "65844", "65855", "65860", "65875", "65879", "65886", "65887", "65888", "65901", "65902", "65913", "65919", "65932", "65934", "65937", "65938", "65939", "65944", "65950", "65956", "65964", "65977", "65983", "65990", "65992", "65993", "65996", "65997", "66015", "66016", "66018", "66020", "66022", "66025", "66035", "66040", "66046", "66047", "66048", "66049", "66054", "66070", "66077", "66082", "66094", "66095", "66099", "66101", "66105", "66108", "66110", "66115", "66117", "66118", "66119", "66121", "66126", "66128", "66133", "66134", "66135", "66137", "66149", "66152", "66154", "66166", "66172", "66174", "66176", "66179", "66181", "66187", "66189", "66190", "66191", "66199", "66210", "66212", "66214", "66222", "66233", "66235", "66238", "66239", "66241", "66242", "66257", "66265", "66271", "66278", "66289", "66294", "66296", "66297", "66303", "66307", "66324", "66335", "66339", "66340", "66343", "66352", "66357", "66368", "66373", "66376", "66377", "66378", "66379", "66392", "66395", "66398", "66399", "66401", "66403", "66407", "66411", "66431", "66438", "66439", "66440", "66445", "66448", "66458", "66459", "66465", "66466", "66468", "66469", "66470", "66471", "66487", "66488", "66490", "66491", "66494", "66496", "66500", "66511", "66526", "66528", "66530", "66531", "66535", "66542", "66570", "66574", "66578", "66579", "66589", "66592", "66601", "66606", "66626", "66637", "66645", "66646", "66657", "66662", "66680", "66681", "66685", "66693", "66695", "66713", "66714", "66721", "66724", "66744", "66745", "66750", "66752", "66753", "66757", "66770", "66775", "66777", "66782", "66783", "66786", "66793", "66794", "66796", "66804", "66807", "66808", "66809", "66817", "66818", "66819", "66842", "66850", "66856", "66864", "66868", "66881", "66886", "66888", "66889", "66893", "66901", "66902", "66903", "66919", "66923", "66935", "66936", "66937", "66940", "66941", "66945", "66948", "66961", "66974", "66976", "66989", "66992", "66993", "67003", "67005", "67007", "67012", "67013", "67021", "67022", "67037", "67040", "67052", "67055", "67066", "67074", "67076", "67078", "67083", "67084", "67085", "67093", "67096", "67108", "67118", "67127", "67143", "67159", "67171", "67181", "67188", "67213", "67215", "67221", "67222", "67223", "67224", "67225", "67233", "67236", "67239", "67247", "67249", "67262", "67263", "67271", "67275", "67280", "67283", "67286", "67288", "67295", "67296", "67298", "67305", "67306", "67308", "67309", "67322", "67323", "67327", "67338", "67340", "67368", "67377", "67388", "67400", "67409", "67421", "67425", "67431", "67433", "67436", "67440", "67451", "67452", "67454", "67473", "67474", "67475", "67477", "67478", "67484", "67490", "67491", "67509", "67521", "67524", "67531", "67532", "67540", "67541", "67553", "67555", "67573", "67580", "67597", "67600", "67605", "67609", "67614", "67629", "67642", "67657", "67662", "67663", "67673", "67685", "67686", "67692", "67693", "67708", "67710", "67711", "67712", "67738", "67739", "67746", "67747", "67754", "67756", "67766", "67770", "67781", "67784", "67785", "67789", "67804", "67806", "67808", "67859", "67862", "67868", "67871", "67877", "67878", "67894", "67895", "67901", "67915", "67932", "67949", "67966", "67983", "67985", "67986", "67993", "67997", "68010", "68012", "68015", "68018", "68020", "68023", "68034", "68050", "68051", "68054", "68055", "68058", "68064", "68069", "68070", "68073", "68081", "68082", "68088", "68089", "68096", "68096", "68100", "68112", "68114", "68119", "68122", "68128", "68135", "68148", "68155", "68158", "68159", "68181", "68184", "68189", "68198", "68205", "68210", "68212", "68213", "68225", "68250", "68260", "68263", "68265", "68269", "68280", "68284", "68287", "68289", "68299", "68306", "68312", "68315", "68321", "68322", "68325", "68329", "68340", "68341", "68344", "68349", "68352", "68354", "68356", "68359", "68363", "68366", "68367", "68368", "68373", "68378", "68380", "68384", "68391", "68401", "68415", "68417", "68439", "68444", "68445", "68451", "68465", "68469", "68471", "68488", "68489", "68493", "68495", "68506", "68507", "68511", "68512", "68528", "68529", "68530", "68534", "68535", "68542", "68545", "68547", "68548", "68556", "68565", "68567", "68568", "68569", "68570", "68571", "68611", "68619", "68626", "68630", "68660", "68669", "68675", "68678", "68682", "68686", "68688", "68694", "68698", "68704", "68706", "68708", "68709", "68710", "68721", "68723", "68735", "68736", "68745", "68746", "68753", "68755", "68762", "68767", "68774", "68777", "68780", "68783", "68791", "68796", "68797", "68799", "68812", "68813", "68817", "68819", "68820", "68822", "68844", "68845", "68849", "68854", "68855", "68858", "68861", "68870", "68872", "68876", "68878", "68882", "68897", "68913", "68937", "68942", "68948", "68950", "68955", "68957", "68962", "68972", "68984", "68986", "68995", "68998", "69002", "69010", "69012", "69013", "69016", "69020", "69021", "69025", "69027", "69030", "69047", "69051", "69058", "69063", "69074", "69084", "69087", "69092", "69095", "69096", "69098", "69105", "69117", "69121", "69126", "69130", "69138", "69139", "69140", "69163", "69168", "69170", "69174", "69196", "69198", "69201", "69207", "69209", "69211", "69212", "69214", "69234", "69238", "69240", "69245", "69257", "69258", "69259", "69261", "69263", "69267", "69270", "69271", "69273", "69274", "69276", "69277", "69279", "69283", "69293", "69300", "69305", "69306", "69310", "69320", "69326", "69327", "69329", "69335", "69348", "69351", "69353", "69375", "69377", "69379", "69382", "69383", "69384", "69393", "69403", "69407", "69408", "69414", "69416", "69417", "69419", "69423", "69429", "69431", "69437", "69439", "69444", "69445", "69459", "69467", "69474", "69476", "69478", "69481", "69495", "69496", "69507", "69509", "69512", "69517", "69525", "69527", "69530", "69533", "69544", "69547", "69551", "69552", "69554", "69558", "69560", "69564", "69566", "69567", "69571", "69572", "69578", "69582", "69583", "69587", "69591", "69592", "69596", "69597", "69598", "69599", "69617", "69621", "69629", "69631", "69633", "69640", "69641", "69643", "69648", "69663", "69672", "69680", "69684", "69695", "69706", "69712", "69717", "69719", "69720", "69728", "69730", "69734", "69735", "69737", "69738", "69740", "69741", "69744", "69745", "69750", "69751", "69752", "69753", "69757", "69764", "69772", "69777", "69778", "69783", "69785", "69786", "69796", "69798", "69802", "69812", "69828", "69902", "69946", "69958", "70019", "70026", "70045", "70063", "70092", "70159", "70191", "70227", "70265", "70272", "70313", "70395", "70538", "70584", "70609", "70714", "70716", "70800", "70841", "70935", "70939", "70972", "70973", "70981", "71035", "71061", "71064", "71069", "71073", "71160", "71210", "71290", "71323", "71344", "71358", "71360", "71453", "71470", "71503", "71534", "71549", "71573", "71598", "71623", "71634", "71652", "71675", "71684", "71686", "71689", "71749", "71807", "71854", "71991", "72004", "72011", "72028", "72142", "72159", "72189", "72324", "72384", "72482", "72500", "72504", "72540", "72542", "72546", "72549", "72566", "72606", "72657", "72722", "72739", "72745", "72797", "72831", "72849", "72856", "72914", "72927", "72934", "72969", "73032", "73074", "73127", "73148", "73169", "73175", "73203", "73206", "73244", "73260", "73264", "73266", "73309", "73312", "73351", "73354", "73381", "73404", "73415", "73435", "73453", "73497", "73521", "73533", "73548", "73564", "73575", "73604", "73627", "73641", "73696", "73741", "73760", "73798", "73801", "73821", "73864", "73885", "73911", "73986", "73995", "73996", "74010", "74043", "74046", "74048", "74053", "74057", "74115", "74149", "74170", "74173", "74180", "74183", "74184", "74191", "74198", "74200", "74202", "74208", "74211", "74212", "74213", "74216", "74229", "74240", "74249", "74250", "74280", "74299", "74307", "74315", "74317", "74318", "74330", "74345", "74347", "74349", "74353", "74364", "74385", "74395", "74405", "74410", "74411", "74451", "74454", "74458", "74466", "74467", "74479", "74489", "74490", "74503", "74511", "74529", "74536", "74549", "74551", "74563", "74566", "74573", "74585", "74592", "74597", "74599", "74603", "74605", "74610", "74614", "74625", "74638", "74639", "74640", "74648", "74658", "74662", "74665", "74677", "74680", "74684", "74691", "74692", "74695", "74701", "74713", "74716", "74741", "74756", "74761", "74773", "74779", "74793", "74803", "74813", "74827", "74828", "74829", "74860", "74865", "74871", "74879", "74908", "74909", "74911", "74921", "74936", "74947", "74948", "74962", "74977", "74979", "74990", "75006", "75013", "75045", "75049", "75058", "75075", "75081", "75091", "75096", "75108", "75113", "75131", "75133", "75137", "75142", "75149", "75156", "75164", "75191", "75199", "75210", "75215", "75216", "75218", "75221", "75225", "75226", "75230", "75238", "75239", "75242", "75245", "75256", "75271", "75280", "75286", "75300", "75304", "75311", "75312", "75313", "75314", "75315", "75330", "75335", "75342", "75348", "75361", "75362", "75363", "75364", "75365", "75366", "75367", "75368", "75369", "75370", "75371", "75372", "75373", "75374", "75375", "75378", "75380", "75383", "75400", "75402", "75407", "75416", "75428", "75432", "75451", "75467", "75470", "75475", "75491", "75492", "75496", "75501", "75507", "75517", "75521", "75525", "75535", "75542", "75548", "75549", "75552", "75556", "75557", "75559", "75561", "75569", "75633", "75637", "75645", "75648", "75649", "75650", "75651", "75653", "75654", "75655", "75666", "75669", "75670", "75676", "75677", "75678", "75682", "75684", "75696", "75711", "75714", "75717", "75718", "75719", "75731", "75736", "75742", "75744", "75745", "75749", "75750", "75754", "75760", "75778", "75779", "75780", "75781", "75785", "75827", "75853", "75854", "75860", "75862", "75880", "75885", "75894", "75902", "75904", "75919", "75922", "75942", "75958", "75979", "75985", "75987", "75991", "75992", "76000", "76001", "76006", "76010", "76012", "76029", "76030", "76052", "76100", "76112", "76132", "76146", "76150", "76153", "76157", "76167", "76172", "76175", "76191", "76199", "76200", "76201", "76202", "76205", "76206", "76215", "76239", "76246", "76253", "76262", "76320", "76335", "76376", "76378", "76417", "76431", "76444", "76445", "76465", "76487", "76497", "76502", "76516", "76520", "76523", "76529", "76535", "76547", "76569", "76573", "76612", "76622", "76652", "76664", "76676", "76681", "76683", "76721", "76727", "76731", "76747", "76748", "76756", "76763", "76773", "76775", "76776", "76777", "76779", "76789", "76800", "76819", "76828", "76865", "76880", "76888", "76894", "76919", "76920", "76923", "76924", "76925", "76926", "76928", "76929", "76932", "76933", "76934", "76935", "76937", "76938", "76940", "76941", "76942", "76943", "76948", "76949", "76951", "76952", "76953", "76955", "76956", "76957", "76958", "76959", "76982", "76984", "76985", "76987", "76988", "76989", "76991", "76992", "76993", "76994", "76996", "76998", "77017", "77033", "77037", "77039", "77055", "77072", "77080", "77081", "77097", "77105", "77106", "77122", "77157", "77186", "77187", "77188", "77198", "77203", "77215", "77216", "77217", "77246", "77248", "77249", "77251", "77293", "77320", "77321", "77329", "77336", "77340", "77343", "77346", "77355", "77388", "77399", "77400", "77433", "77438", "77458", "77464", "77469", "77500", "77501", "77511", "77519", "77530", "77533", "77548", "77560", "77565", "77567", "77587", "77595", "77596", "77602", "77603", "77617", "77628", "77653", "77657", "77667", "77668", "77689", "77702", "77703", "77715", "77720", "77727", "77729", "77730", "77743", "77745", "77751", "77762", "77766", "77767", "77800", "77802", "77803", "77804", "77809", "77813", "77816", "77818", "77827", "77835", "77839", "77841", "77842", "77851", "77855", "77861", "77863", "77866", "77869", "77878", "77884", "77897", "77900", "77906", "77907", "77915", "77918", "77919", "77921", "77922", "77923", "77924", "77930", "77934", "77936", "77941", "77942", "77943", "77945", "77949", "77957", "77962", "77965", "77970", "77976", "77988", "77990", "77991", "77993", "77994", "77996", "78011", "78014", "78016", "78040", "78041", "78043", "78044", "78050", "78055", "78057", "78064", "78066", "78069", "78078", "78080", "78086", "78088", "78090", "78106", "78110", "78129", "78144", "78147", "78148", "78153", "78162", "78170", "78175", "78188", "78189", "78193", "78203", "78206", "78229", "78232", "78234", "78238", "78263", "78264", "78265", "78274", "78287", "78291", "78295", "78320", "78321", "78328", "78340", "78345", "78355", "78358", "78363", "78395", "78403", "78413", "78420", "78422", "78431", "78453", "78470", "78479", "78483", "78499", "78520", "78533", "78543", "78563", "78574", "78605", "78609", "78617", "78619", "78640", "78646", "78647", "78650", "78652", "78655", "78661", "78685", "78693", "78708", "78716", "78720", "78739", "78743", "78749", "78761", "78780", "78781", "78784", "78798", "78807", "78813", "78816", "78825", "78826", "78832", "78848", "78850", "78858", "78870", "78891", "78895", "78896", "78903", "78908", "78921", "78937", "78962", "78970", "78977", "78983", "78994", "79017", "79021", "79023", "79027", "79032", "79043", "79049", "79073", "79075", "79077", "79080", "79081", "79092", "79093", "79098", "79111", "79129", "79142", "79143", "79152", "79165", "79171", "79173", "79183", "79184", "79185", "79200", "79212", "79213", "79214", "79217", "79221", "79234", "79235", "79253", "79254", "79261", "79262", "79263", "79268", "79278", "79286", "79322", "79334", "79339", "79355", "79362", "79365", "79390", "79409", "79416", "79421", "79422", "79424", "79425", "79426", "79435", "79444", "79451", "79462", "79467", "79479", "79492", "79506", "79517", "79531", "79556", "79562", "79572", "79584", "79588", "79589", "79590", "79591", "79605", "79610", "79611", "79615", "79617", "79621", "79643", "79644", "79645", "79646", "79647", "79648", "79657", "79659", "79670", "79674", "79680", "79686", "79701", "79745", "79756", "79762", "79767", "79777", "79778", "79779", "79782", "79783", "79788", "79790", "79793", "79797", "79803", "79804", "79809", "79912", "79916", "79941", "79965", "79989", "80028", "80072", "80082", "80119", "80138", "80231", "80235", "80244", "80246", "80250", "80251", "80266", "80268", "80273", "80279", "80291", "80296", "80308", "80315", "80317", "80318", "80346", "80348", "80360", "80423", "80443", "80471", "80474", "80601", "80623", "80672", "80690", "80698", "80709", "80751", "80757", "80785", "80786", "80813", "80821", "80822", "80829", "80838", "80869", "80876", "80916", "80924", "80937", "80952", "80967", "80985", "81037", "81053", "81089", "81127", "81137", "81138", "81163", "81195", "81205", "81260", "81262", "81266", "81269", "81273", "81308", "81315", "81321", "81323", "81329", "81330", "81360", "81366", "81399", "81409", "81419", "81424", "81432", "81446", "81452", "81465", "81474", "81488", "81513", "81555", "81560", "81570", "81602", "81610", "81620", "81624", "81630", "81670", "81674", "81676", "81690", "81691", "81692", "81700", "81704", "81708", "81710", "81715", "81716", "81720", "81721", "81723", "81726", "81729", "81757", "81758", "81770", "81775", "81782", "81787", "81796", "81824", "81836", "81847", "81868", "81883", "81888", "81897", "81905", "81906", "81929", "81934", "81965", "81969", "81972", "82009", "82028", "82048", "82051", "82064", "82075", "82094", "82224", "82235", "82265", "82270", "82279", "82294", "82295", "82305", "82313", "82328", "82335", "82351", "82374", "82384", "82385", "82391", "82406", "82412", "82423", "82440", "82463", "82465", "82482", "82489", "82490", "82492", "82527", "82532", "82546", "82555", "82558", "82570", "82590", "82622", "82626", "82631", "82635", "82656", "82676", "82688", "82689", "82718", "82720", "82754", "82800", "82806", "82814", "82821", "82826", "82835", "82858", "82865", "82866", "82882", "82892", "82900", "82919", "82924", "82938", "82939", "82953", "82963", "82977", "82979", "82983", "83002", "83030", "83031", "83041", "83046", "83059", "83062", "83065", "83068", "83082", "83087", "83093", "83096", "83097", "83105", "83110", "83114", "83116", "83119", "83129", "83137", "83140", "83144", "83150", "83160", "83273", "83313", "83316", "83342", "83363", "83367", "83464", "83520", "83530", "83622", "83634", "83640", "83652", "83662", "83771", "83790", "83824", "83833", "83862", "83948", "84145", "84186", "84226", "84300", "84350", "84374", "84448", "84483", "84496", "84497", "84508", "84555", "84653", "84727", "84731", "84789", "84808", "84893", "84936", "84971", "85002", "85037", "85063", "85098", "85099", "85264", "85271", "85364", "85408", "85440", "85445", "85459", "85517", "85539", "85566", "85618", "85645", "85662", "85680", "85698", "85751", "85815", "85828", "85869", "85993", "86073", "86075", "86118", "86163", "86187", "86226", "86318", "86351", "86401", "86465", "86483", "86486", "86490", "86503", "86507", "86521", "86537", "86561", "86583", "86590", "86632", "86642", "86653", "86663", "86680", "86718", "86720", "86722", "86784", "86786", "86823", "86829", "86858", "87183", "87210", "87262", "87619", "87862", "87893", "88048", "88364", "88644", "88681", "88858", "89347", "89380", "89594", "90124", "90161", "90263", "90272", "90280", "90288", "90327", "90336", "90396", "90399", "90402", "90411", "90414", "90417", "90421", "90425", "90438", "90441", "90452", "90476", "90488", "90491", "90498", "90499", "90690", "90820", "90918", "91160", "91240", "91314", "91339", "91380", "91404", "91405", "91406", "91411", "91418", "91434", "91435", "91502", "91606", "91709", "91737", "91777", "91780", "91877", "91890", "91896", "91919", "91921", "91946", "91954", "91972", "91993", "92021", "92024", "92045", "92060", "92103", "92123", "92163", "92173", "92174", "92192", "92242", "92264", "92276", "92282", "92283", "92291", "92362", "92397", "92400", "92421", "92439", "92453", "92468", "92476", "92499", "92559", "92560", "92574", "92617", "92711", "92738", "92762", "92769", "92777", "92779", "92781", "92848", "92861", "92868", "92915", "92921", "92930", "92954", "92984", "93055", "93119", "93126", "93132", "93139", "93182", "93235", "93245", "93262", "93337", "93350", "93384", "93394", "93398", "93407", "93506", "93541", "93577", "93596", "93622", "93629", "93640", "93681", "93689", "93698", "93727", "93736", "93744", "93795", "93825", "93827", "93859", "93860", "93861", "93869", "93902", "93936", "93952", "93969", "93972", "93980", "93982", "93985", "94012", "94033", "94037", "94042", "94050", "94073", "94096", "94096", "94100", "94111", "94114", "94120", "94123", "94134", "94135", "94136", "94150", "94151", "94154", "94155", "94161", "94165", "94168", "94172", "94180", "94184", "94194", "94197", "94204", "94212", "94218", "94220", "94223", "94237", "94238", "94239", "94243", "94245", "94248", "94252", "94255", "94259", "94266", "94272", "94273", "94275", "94281", "94285", "94287", "94297", "94301", "94304", "94310", "94311", "94321", "94325", "94330", "94332", "94339", "94340", "94341", "94350", "94353", "94361", "94365", "94367", "94369", "94373", "94377", "94384", "94385", "94386", "94387", "94395", "94397", "94422", "94432", "94433", "94436", "94437", "94440", "94445", "94450", "94463", "94467", "94482", "94485", "94489", "94493", "94495", "94500", "94509", "94513", "94518", "94520", "94528", "94532", "94546", "94547", "94550", "94556", "94561", "94563", "94572", "94579", "94584", "94586", "94587", "94598", "94602", "94606", "94607", "94619", "94622", "94636", "94638", "94639", "94642", "94644", "94645", "94648", "94655", "94657", "94662", "94671", "94675", "94698", "94767", "94776", "94834", "94895", "94909", "95015", "95085", "95134", "95137", "95140", "95156", "95178", "95184", "95208", "95220", "95244", "95250", "95256", "95272", "95273", "95274", "95306", "95313", "95346", "95347", "95426", "95453", "95460", "95483", "95540", "95544", "95551", "95571", "95573", "95580", "95585", "95591", "95592", "95602", "95634", "95653", "95681", "95685", "95688", "95693", "95753", "95760", "95768", "95789", "95806", "95866", "95869", "95897", "95906", "95909", "95933", "95958", "96022", "96027", "96053", "96097", "96125", "96133", "96194", "96360", "96368", "96452", "96468", "96468", "96490", "96508", "96513", "96523", "96533", "96547", "96548", "96576", "96671", "96678", "96686", "96701", "96703", "96770", "96903", "96906", "96918", "96934", "96937", "96985", "96987", "96994", "97013", "97113", "97124", "97201", "97217", "97366", "97376", "97450", "97459", "97474", "97535", "97564", "97576", "97581", "97584", "97638", "97675", "97690", "97691", "97715", "97733", "97744", "97785", "97811", "97814", "97825", "97853", "97896", "97897", "97947", "97963", "98042", "98049", "98101", "98132", "98159", "98166", "98189", "98226", "98294", "98295", "98328", "98330", "98341", "98356", "98366", "98369", "98399", "98414", "98435", "98438", "98457", "98457", "98471", "98598", "98609", "98618", "98639", "98675", "98724", "98752", "98758", "98785", "98808", "98844", "98873", "98974", "98978", "98999", "99054", "99062", "99111", "99147", "99155", "99216", "99223", "99236", "99256", "99279", "99289", "99294", "99318", "99324", "99384", "99401", "99419", "99453", "99488", "99500", "99520", "99568", "99592", "99614", "99616", "99641", "99755", "99780", "99807", "99810", "99838", "99851", "99865", "99871", "99916", "99927", "99928", "99960", "99961", "99976", "99987", "99989", "100040", "100123", "100129", "100133", "100137", "100139", "100140", "100144", "100167", "100172", "100182", "100219", "100233", "100234", "100235", "100265", "100320", "100327", "100328", "100400", "100407", "100410", "100417", "100419", "100422", "100436", "100465", "100535", "100558", "100565", "100610", "100640", "100658", "100660", "100678", "100680", "100686", "100695", "100715", "100717", "100719", "100767", "100773", "100776", "100818", "100825", "100846", "100854", "100866", "100882", "100895", "100906", "100923", "100934", "101001", "101003", "101007", "101011", "101015", "101034", "101067", "101072", "101084", "101112", "101119", "101145", "101153", "101161", "101217", "101250", "101264", "101283", "101292", "101296", "101442", "101451", "101468", "101505", "101514", "101543", "101575", "101620", "101642", "101644", "101681", "101683", "101728", "101732", "101734", "101739", "101748", "101749", "101760", "101762", "101764", "101768", "101770", "101772", "101778", "101780", "101782", "101787", "101789", "101796", "101798", "101805", "101806", "101808", "101815", "101821", "101824", "101834", "101835", "101839", "101840", "101841", "101842", "101843", "101847", "101848", "101852", "101855", "101857", "101858", "101866", "101872", "101874", "101875", "101879", "101888", "101890", "101891", "101893", "101898", "101899", "101901", "101905", "101907", "101908", "101909", "101922", "101927", "101930", "101932", "101933", "101934", "101941", "101942", "101944", "101952", "101955", "101970", "101972", "101977", "101978", "101981", "101983", "101986", "101988", "101990", "101994", "101996", "101997", "102000", "102004", "102015", "102020", "102029", "102032", "102034", "102039", "102040", "102044", "102046", "102054", "102055", "102059", "102064", "102066", "102079", "102081", "102083", "102085", "102086", "102092", "102093", "102100", "102104", "102112", "102118", "102123", "102125", "102129", "102131", "102134", "102140", "102142", "102145", "102147", "102152", "102154", "102157", "102171", "102174", "102179", "102180", "102201", "102202", "102205", "102206", "102209", "102221", "102228", "102232", "102286", "102344", "102348", "102410", "102430", "102431", "102479", "102482", "102496", "102497", "102524", "102548", "102559", "102590", "102639", "102672", "102681", "102684", "102706", "102734", "102789", "102794", "102889", "102929", "102933", "102953", "102962", "102994", "103001", "103007", "103114", "103115", "103141", "103144", "103216", "103268", "103272", "103362", "103386", "103392", "103443", "103464", "103465", "103470", "103648", "103707", "103755", "103786", "103791", "103834", "103938", "104000", "104047", "104109", "104110", "104118", "104186", "104214", "104216", "104235", "104270", "104280", "104319", "104377", "104444", "104478", "104525", "104541", "104560", "104621", "104667", "104733", "104783", "104815", "104844", "104846", "104851", "104857", "104895", "104920", "104979", "105082", "105089", "105092", "105187", "105198", "105206", "105229", "105256", "105265", "105273", "105310", "105340", "105345", "105352", "105356", "105384", "105397", "105444", "105466", "105519", "105537", "105578", "105579", "105689", "105742", "105744", "105748", "105761", "105772", "105781", "105783", "105786", "105802", "105806", "105816", "105821", "105822", "105826", "105831", "105838", "105844", "105845", "105863", "105868", "105878", "105883", "105891", "105911", "105915", "105916", "105927", "105928", "105929", "105953", "105954", "105962", "105968", "105969", "105975", "105978", "105979", "105982", "105987", "105989", "106004", "106017", "106023", "106027", "106032", "106033", "106040", "106043", "106053", "106060", "106062", "106064", "106083", "106092", "106113", "106117", "106123", "106128", "106134", "106154", "106158", "106188", "106205", "106210", "106217", "106220", "106227", "106231", "106233", "106240", "106243", "106252", "106267", "106279", "106292", "106302", "106309", "106315", "106327", "106328", "106339", "106342", "106351", "106354", "106359", "106369", "106382", "106388", "106396", "106399", "106411", "106426", "106427", "106430", "106432", "106440", "106441", "106442", "106444", "106446", "106475", "106487", "106492", "106494", "106496", "106501", "106511", "106532", "106535", "106542", "106544", "106550", "106558", "106562", "106567", "106571", "106579", "106585", "106586", "106591", "106594", "106596", "106597", "106598", "106608", "106609", "106625", "106638", "106644", "106657", "106674", "106675", "106676", "106681", "106682", "106684", "106699", "106716", "106720", "106854", "106973", "106974", "107003", "107068", "107082", "107094", "107127", "107153", "107168", "107190", "107197", "107247", "107268", "107342", "107438", "107449", "107479", "107484", "107496", "107513"]
}

print(f"Loaded {len(ground_truth)} queries for evaluation.")

Loaded 3 queries for evaluation.


## 4. Execution & Analysis

In [18]:
def compare_retrievers(query, relevant_ids, k=5):
    results = []
    
    retrievers = {
        "Dense": dense_retriever,
        "Hybrid": hybrid_retriever,
        "Ensemble": ensemble_retriever
    }
    
    print(f"\n=== Query: {query} ===")
    print(f"Relevant IDs ({len(relevant_ids)}): {relevant_ids}")
    
    row_data = {"Query": query}
    
    for name, retriever in retrievers.items():
        try:
            # Retrieve documents
            docs = retriever.invoke(query)
            
            # Extract content_id as string for comparison
            retrieved_ids = [str(doc.metadata.get("no")) for doc in docs]
            
            # Calculate Metrics using Sklearn
            recall, ndcg = calculate_metrics_using_sklearn(retrieved_ids, relevant_ids, k=k)
            
            row_data[f"{name}_Recall@{k}"] = recall
            row_data[f"{name}_NDCG@{k}"] = ndcg
            
            print(f"[{name}] Recall@{k}: {recall:.4f}, NDCG@{k}: {ndcg:.4f}")
            # Show top 3 results for visual check
            for i, doc in enumerate(docs[:3]):
                print(f"  {i+1}. {doc.page_content} (ID: {doc.metadata.get('no')})")
        except Exception as e:
            print(f"[{name}] Error: {e}")
            row_data[f"{name}_Recall@{k}"] = None
            row_data[f"{name}_NDCG@{k}"] = None
             
    return row_data

# Run Comparison
evaluation_results = []
K_METRIC = 5 # Recall@5, NDCG@5

for query, relevant_ids in ground_truth.items():
    # relevant_ids를 문자열로 변환하여 처리 (DB 타입 일치를 위해)
    relevant_ids_str = [str(id) for id in relevant_ids]
    
    if not relevant_ids_str:
        print(f"\nSkipping query '{query}' due to empty ground truth.")
        continue
        
    result = compare_retrievers(query, relevant_ids_str, k=K_METRIC)
    evaluation_results.append(result)

if evaluation_results:
    df = pd.DataFrame(evaluation_results)
    print("\n=== Final Results Table ===")
    display(df)
else:
    print("\nNo successful evaluations completed. Please check ground_truth data.")


=== Query: 야경이 이쁜 곳 ===
Relevant IDs (384): ['56696', '56745', '56755', '56777', '56816', '56834', '56835', '56847', '56888', '56890', '56895', '56977', '57052', '57142', '57236', '57254', '57263', '57268', '57345', '57369', '57402', '57456', '57461', '57527', '57573', '57605', '57640', '57669', '57691', '57734', '57737', '57757', '57798', '57825', '57892', '57904', '57941', '57965', '58004', '58009', '58031', '58081', '58094', '58209', '58296', '58298', '58311', '58338', '58486', '58606', '58619', '58660', '58665', '58666', '58678', '58686', '58693', '58711', '58724', '58747', '58750', '58754', '58761', '58764', '58774', '58783', '58802', '58817', '58823', '58830', '58840', '58842', '58859', '58884', '58887', '58899', '58904', '58915', '58932', '58946', '58947', '58986', '59036', '59043', '59135', '59148', '59165', '59861', '59883', '60698', '60855', '62014', '62636', '62661', '63124', '63223', '63344', '63519', '63683', '63859', '64035', '64112', '64600', '64846', '65131', '65190', 

,Query,Dense_Recall@5,Dense_NDCG@5,Hybrid_Recall@5,Hybrid_NDCG@5,Ensemble_Recall@5,Ensemble_NDCG@5
0,야경이 이쁜 곳,0.013021,1.000000,0.005208,0.360055,0.013021,1.000000
1,아이와 함께 가기 좋은 곳,0.031579,0.654809,0.021053,0.345191,0.031579,0.654809
2,산책하기 좋은 곳,0.001045,1.000000,0.000627,0.654809,0.001045,1.000000
